In [1]:
import pyautogui
import time
import random
from PIL import ImageGrab

# ==========================================
# 混合動力版設定 (Hybrid Mode)
# ==========================================
pyautogui.FAILSAFE = True
pyautogui.PAUSE = 0.005  # 快速點擊

TOP_LEFT = (327, 427)
BOTTOM_RIGHT = (608, 706)
GRID_SIZE = 8

def get_grid_coords():
    start_x, start_y = TOP_LEFT
    end_x, end_y = BOTTOM_RIGHT
    step_x = (end_x - start_x) / (GRID_SIZE - 1)
    step_y = (end_y - start_y) / (GRID_SIZE - 1)
    grid = []
    for row in range(GRID_SIZE):
        row_coords = []
        for col in range(GRID_SIZE):
            x = int(start_x + col * step_x)
            y = int(start_y + row * step_y)
            row_coords.append((x, y))
        grid.append(row_coords)
    return grid

def get_fast_stable_color(screenshot, x, y):
    """
    混合採樣：採樣中心周圍 9 個點，兼顧速度與穩定性。
    """
    points = [(x,y), (x-2,y), (x+2,y), (x,y-2), (x,y+2), (x-2,y-2), (x+2,y+2), (x-2,y+2), (x+2,y-2)]
    r_t, g_t, b_t = 0, 0, 0
    for px, py in points:
        r, g, b = screenshot.getpixel((px, py))
        r_t += r; g_t += g; b_t += b
    return (r_t // 9, g_t // 9, b_t // 9)

def identify_gem(color):
    r, g, b = color
    # 使用您提供的精確數值範圍
    if r > 170 and g < 110 and b < 110: return 1 # 紅
    if r > 180 and 50 < g < 150 and b < 70: return 2 # 橙
    if r > 170 and g > 170 and b < 110: return 3 # 黃
    if r < 130 and g > 110 and b < 160: return 4 # 綠
    if r < 100 and 90 < g < 190 and b > 170: return 5 # 藍
    if r > 170 and g < 100 and b > 170: return 6 # 紫
    if r > 190 and g > 190 and b > 190: return 7 # 白
    return 0

def get_board_state(grid):
    screenshot = ImageGrab.grab()
    board = []
    for row in range(GRID_SIZE):
        board_row = []
        for col in range(GRID_SIZE):
            x, y = grid[row][col]
            avg_color = get_fast_stable_color(screenshot, x, y)
            board_row.append(identify_gem(avg_color))
        board.append(board_row)
    return board

def is_match(board, r, c):
    gem = board[r][c]
    if gem == 0: return False
    # 橫向
    count = 1
    i = c - 1
    while i >= 0 and board[r][i] == gem: count += 1; i -= 1
    i = c + 1
    while i < GRID_SIZE and board[r][i] == gem: count += 1; i += 1
    if count >= 3: return True
    # 縱向
    count = 1
    i = r - 1
    while i >= 0 and board[i][c] == gem: count += 1; i -= 1
    i = r + 1
    while i < GRID_SIZE and board[i][c] == gem: count += 1; i += 1
    return count >= 3

def check_swap(board, r1, c1, r2, c2):
    if board[r1][c1] == 0 or board[r2][c2] == 0: return False
    if board[r1][c1] == board[r2][c2]: return False
    temp = [row[:] for row in board]
    temp[r1][c1], temp[r2][c2] = temp[r2][c2], temp[r1][c1]
    return is_match(temp, r1, c1) or is_match(temp, r2, c2)

def find_moves(board):
    moves = []
    for r in range(GRID_SIZE):
        for c in range(GRID_SIZE):
            if c < GRID_SIZE - 1 and check_swap(board, r, c, r, c + 1):
                moves.append(((r, c), (r, c + 1)))
            if r < GRID_SIZE - 1 and check_swap(board, r, c, r + 1, c):
                moves.append(((r, c), (r + 1, c)))
    return moves

def main():
    print("--- Bejeweled Blitz 混合動力版 (高分+防卡) ---")
    print("腳本將在 5 秒後開始...")
    time.sleep(5)
    grid = get_grid_coords()
    
    no_move_count = 0
    
    while True:
        try:
            board = get_board_state(grid)
            moves = find_moves(board)
            
            if moves:
                # 執行所有發現的移動，但每個移動間不加長等待
                for move in moves:
                    (r1, c1), (r2, c2) = move
                    pyautogui.click(grid[r1][c1][0], grid[r1][c1][1])
                    pyautogui.click(grid[r2][c2][0], grid[r2][c2][1])
                no_move_count = 0
            else:
                no_move_count += 1
            
            # 如果連續找不到移動，立即執行擾動，不給遊戲卡住的機會
            if no_move_count >= 2:
                # 執行更積極的擾動
                for _ in range(2):
                    r = random.randint(0, GRID_SIZE - 1)
                    c = random.randint(0, GRID_SIZE - 2)
                    pyautogui.click(grid[r][c][0], grid[r][c][1])
                    pyautogui.click(grid[r][c+1][0], grid[r][c+1][1])
                no_move_count = 0
                
            # 極短暫的循環間隔，保持高頻掃描
            time.sleep(0.01)
                
        except pyautogui.FailSafeException:
            print("\n腳本已停止。")
            break
        except Exception:
            pass

if __name__ == "__main__":
    main()


--- Bejeweled Blitz 混合動力版 (高分+防卡) ---
腳本將在 5 秒後開始...

腳本已停止。
